In [ ]:
!pip install kaggle
!mkdir -p ~/.kaggle
upload_file_path = '/content/kaggle.json'
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
from huggingface_hub import login
from google.colab import userdata

secret_label = "HFHub"
secret_value = userdata.get(secret_label)
login(token=secret_value)

In [1]:
%pip install \
    datasets \
    evaluate \
    rouge_score\
    loralib \
    evaluate \
    accelerate \
    bitsandbytes \
    trl \
    peft \
    -U --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Note: Remove this cell if using datasets>=4.4.1 which requires pyarrow>=21.0.0
# The pyarrow downgrade causes compatibility issues
# If you need pyarrow 19.0.0 for other reasons, pin datasets version:
# !pip install datasets==3.0.0 pyarrow==19.0.0

# For most cases, just use the latest compatible versions:
# !pip install pyarrow>=21.0.0

In [1]:
!pip install tf-keras

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 2.0 MB/s eta 0:00:01
   ------------------------ --------------- 1.0/1.7 MB 2.3 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 1.9 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 1.6 MB/s  0:00:01


In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import pandas as pd
import re
import numpy as np
import string
from nltk.corpus import stopwords
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfTransformer,TfidfVectorizer
from sklearn.pipeline import Pipeline
import evaluate

d:\private\Coding\Aliaa Graduation Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# !pip install -q -U git+https://github.com/huggingface/peft.git

In [3]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

In [5]:
import transformers

torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

In [6]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

compute_dtype = getattr(torch, "float16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    quantization_config=quant_config,
    device_map={"": 0}
)
model.config.use_cache = False
model.config.pretraining_tp = 1

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer_config.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1028)')))"), '(Request ID: a9b67cec-d114-46f6-b621-12002d4fe364)')' thrown while requesting HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
401 Client Error. (Request ID: Root=1-6921b144-681c7a8720b675c02da1cca3;c5326ab8-b94b-488f-b25b-7f665d40b4a6)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [4]:
# Set pad_token as end-of-sentence token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [5]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(model))

trainable model parameters: 1050939392
all model parameters: 4540600320
percentage of trainable model parameters: 23.15%


## <b>3 <span style='color:#9146ff'>|</span> Data Preparation </b>


In [ ]:
import os
import zipfile

# Download dataset
!kaggle datasets download -d omgits0mar/arabic-instruct-chatbot-dataset

# Cross-platform unzip (works on Windows and Linux)
zip_file = 'arabic-instruct-chatbot-dataset.zip'
if os.path.exists(zip_file):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall('.')
    print(f"Extracted {zip_file} successfully")
else:
    print(f"Warning: {zip_file} not found")

In [10]:
import pandas as pd

# Load the Parquet file
df = pd.read_parquet('/content/train-00000-of-00001-10520e8228c2c104.parquet')

# Display the first few rows
df.head()


,instruction,input,output
0,أعط ثلاث نصائح للبقاء بصحة جيدة.,,1- تناول نظامًا غذائيًا متوازنًا وتأكد من تنا...
1,ما هي الألوان الثلاثة الأساسية؟,,الألوان الثلاثة الأساسية هي الأحمر والأزرق وا...
2,صف بنية الذرة.,,تتكون الذرة من نواة تحتوي على البروتونات والن...
3,كيف يمكننا تقليل تلوث الهواء؟,,هناك عدد من الطرق للحد من تلوث الهواء ، مثل ال...
4,صف وقتًا كان عليك فيه اتخاذ قرار صعب.,,كان علي أن أتخذ قرارًا صعبًا عندما كنت أعمل كم...


## <b>4 <span style='color:#9146ff'>|</span> Data Preprocessing </b>


In [11]:
# Calculate the maximum length of text in the 'instruction' column
max_length_instruction = df['instruction'].apply(len).mean()

# Calculate the maximum length of text in the 'output' column
max_length_output = df['output'].apply(len).mean()

# Print the results
print(f"Maximum length of 'instruction': {max_length_instruction}")
print(f"Maximum length of 'output': {max_length_output}")

Maximum length of 'instruction': 47.95117495480943
Maximum length of 'output': 233.013211030345


In [12]:
def tokenize_function(row):
    # Tokenize the conversations
    question = ' '.join(row["instruction"]) if isinstance(row["instruction"], list) else row["instruction"]

    row['input_ids'] = tokenizer(question, padding="max_length", truncation=True, max_length = 128, return_tensors="pt").input_ids[0]

    # Assuming "answer" column is already a string, no need for conversion
    row['labels'] = tokenizer(row["output"], padding="max_length", truncation=True, max_length = 256, return_tensors="pt").input_ids[0]

    return row


# Tokenize the DataFrame
tokenized_df = df.apply(tokenize_function, axis=1)

In [13]:
# Convert columns to list
tokenized_df['input_ids'] = tokenized_df['input_ids'].apply(lambda x: x.tolist())
tokenized_df['labels'] = tokenized_df['labels'].apply(lambda x: x.tolist())

In [14]:
tokenized_df

,instruction,input,output,input_ids,labels
0,أعط ثلاث نصائح للبقاء بصحة جيدة.,,1- تناول نظامًا غذائيًا متوازنًا وتأكد من تنا...,"[128000, 106173, 44735, 117075, 118201, 100462...","[128000, 220, 16, 12, 40534, 101537, 73904, 10..."
1,ما هي الألوان الثلاثة الأساسية؟,,الألوان الثلاثة الأساسية هي الأحمر والأزرق وا...,"[128000, 101237, 104380, 100461, 8700, 100539,...","[128000, 100461, 8700, 100539, 102432, 109413,..."
2,صف بنية الذرة.,,تتكون الذرة من نواة تحتوي على البروتونات والن...,"[128000, 104477, 100829, 74541, 102554, 101341...","[128000, 112077, 103967, 102554, 101341, 64337..."
3,كيف يمكننا تقليل تلوث الهواء؟,,هناك عدد من الطرق للحد من تلوث الهواء ، مثل ال...,"[128000, 114804, 106666, 101537, 40534, 101471...","[128000, 108241, 101052, 105300, 64337, 101979..."
4,صف وقتًا كان عليك فيه اتخاذ قرار صعب.,,كان علي أن أتخذ قرارًا صعبًا عندما كنت أعمل كم...,"[128000, 104477, 110521, 101333, 102037, 12741...","[128000, 102087, 104537, 100822, 64515, 14628,..."
...,...,...,...,...,...
51997,قم بإنشاء مثال لما يجب أن ترغب فيه السيرة الذ...,,جين تريمين \ n1234 Main Street، Anytown، CA 98...,"[128000, 117659, 28946, 107078, 118712, 119979...","[128000, 34190, 100327, 40534, 113690, 100327,..."
51998,رتب العناصر الواردة أدناه بالترتيب لإكمال الجم...,كعكة لي الأكل,أنا آكل الكعكة.,"[128000, 11318, 100936, 119424, 110732, 105155...","[128000, 127389, 100281, 102812, 101100, 24102..."
51999,اكتب فقرة تمهيدية عن شخص مشهور.,ميشيل أوباما,ميشيل أوباما امرأة ملهمة ارتقت إلى مستوى التح...,"[128000, 110973, 100936, 119932, 101341, 10170...","[128000, 102606, 33890, 96298, 64515, 100708, ..."
52000,قم بإنشاء قائمة من خمسة أشياء يجب على المرء أ...,,1. ابحث عن الفرص المحتملة وفكر مليًا في الخيار...,"[128000, 117659, 28946, 107078, 118712, 123797...","[128000, 16, 13, 101558, 116246, 100926, 10865..."


In [15]:
from datasets import Dataset

# Assuming `tokenized_df` is your pandas DataFrame
dataset = Dataset.from_pandas(tokenized_df[:10000])

In [16]:
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'input_ids', 'labels'],
    num_rows: 10000
})

In [ ]:
from datasets import Dataset

# Use full dataset for better training (or subset for faster experimentation)
# Using 10000 samples as in original, but you can increase this
dataset_full = Dataset.from_pandas(tokenized_df[:10000])

# Combine 'instruction' and 'output' into a 'text' column for SFTTrainer
dataset_full = dataset_full.map(lambda x: {"text": x["instruction"] + " " + x["output"]}, batched=False)

# Split into train and test sets (90% train, 10% test)
train_test_split = dataset_full.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
#Model Training and Fine-tuning

In [25]:
# Load LoRA configuration
peft_args = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=8,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
# Set training parameters
training_params = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    eval_strategy="steps",  # Enable evaluation during training
    eval_steps=200,  # Evaluate every 200 steps
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    load_best_model_at_end=True,  # Load best model after training
    metric_for_best_model="eval_loss",
)

In [7]:
from peft import get_peft_model, TaskType

peft_model = get_peft_model(model,
                            peft_args)
print(print_number_of_trainable_model_parameters(peft_model))

NameError: name 'model' is not defined

In [49]:
from trl import SFTTrainer
help(SFTTrainer)


Help on class SFTTrainer in module trl.trainer.sft_trainer:

class SFTTrainer(trl.trainer.base_trainer.BaseTrainer)
 |  SFTTrainer(model: Union[str, transformers.modeling_utils.PreTrainedModel], args: Union[trl.trainer.sft_config.SFTConfig, transformers.training_args.TrainingArguments, NoneType] = None, data_collator: Optional[Callable[[list[Any]], dict[str, Any]]] = None, train_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, NoneType] = None, eval_dataset: Union[datasets.arrow_dataset.Dataset, dict[str, datasets.arrow_dataset.Dataset], NoneType] = None, processing_class: Union[transformers.tokenization_utils_base.PreTrainedTokenizerBase, transformers.processing_utils.ProcessorMixin, NoneType] = None, compute_loss_func: Optional[Callable] = None, compute_metrics: Optional[Callable[[transformers.trainer_utils.EvalPrediction], dict]] = None, callbacks: Optional[list[transformers.trainer_callback.TrainerCallback]] = None, optimizers: tuple[typing.

### SFTTrainer:

- Supervised Fine-tuning (SFT): Optimized for fine-tuning pre-trained models with smaller datasets on supervised learning tasks.
- Simpler interface: Provides a streamlined workflow with fewer configuration options, making it easier to get started.
- Efficient memory usage: Uses techniques like parameter-efficient (PEFT) and packing optimizations to reduce memory consumption during training.
- Faster training: Achieves comparable or better accuracy with smaller datasets and shorter training times than Trainer.

In [ ]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_args,
    dataset_text_field="text",
    max_seq_length=512,  # Increased for Arabic text which can be longer
    tokenizer=tokenizer,
    args=training_params,
    packing=False,
)

### Training

In [97]:
trainer.train()

NameError: name 'trainer' is not defined

### Save model & Publish

In [ ]:
trainer.model.save_pretrained("./llama-3-8B-Arabic")
tokenizer.save_pretrained("./llama-3-8B-Arabic")

('./llama-3-8B-Arabic/tokenizer_config.json',
 './llama-3-8B-Arabic/special_tokens_map.json',
 './llama-3-8B-Arabic/tokenizer.json')

In [ ]:
# model.push_to_hub("")
# tokenizer.push_to_hub("")

## <b>6 <span style='color:#9146ff'>|</span> Testing the model performance on a single inference </b>


In [ ]:
def single_inference(question):
    messages = [
        {"role": "system", "content": "اجب علي الاتي بالعربي فقط."},
    ]

    messages.append({"role": "user", "content": question})


    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.4,
    #     top_p=0.9,
    )
    response = outputs[0][input_ids.shape[-1]:]
    output = tokenizer.decode(response, skip_special_tokens=True)
    return output

In [ ]:
question = """ما هي طريقة عمل البيتزا , اجب في خطوات"""

answer = single_inference(question)

print(f'INPUT QUESTION:\n{question}')
print(f'\n\nModel Answer:\n{answer}')

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


INPUT QUESTION:
ما هي طريقة عمل البيتزا , اجب في خطوات


Model Answer:
لإعداد البيتزا ، اتبع الخطوات التالية:

1. قم بإعداد العجينة: قم بجمع 2 كوب من الدقيق ، 1 كوب من الماء ، 1 ملعقة صغيرة من الخميرة ، 1 ملعقة صغيرة من الملح ، و 2 ملعقة صغيرة من الزيت. قم بترسيب العجينة في وعاء ، ثم قم بتغطيتها بالكفاف وتعطيتها وقتًا لترسيبها.
2. قم بترسيب العجينة: قم بترسيب العجينة في وعاء ، ثم قم بتغطيتها بالكفاف وتعطيتها وقتًا لترسيبها.
3. قم بترتيب المواد: قم بترتيب المواد التالية: 1 رطل من الجبن ، 1 رطل من لحم البقر ، 1 رطل من لحم الدجاج ، 1 ملعقة صغيرة من البصل ، 1 ملعقة صغيرة من الخضار ، 1 ملعقة صغيرة من الكمون ، 1 ملعقة صغيرة من الفلفل ، 1


In [ ]:
question = """   """

answer = single_inference(question)

print(f'INPUT QUESTION:\n{question}')
print(f'\n\nModel Answer:\n{answer}')